6/8/26. Trying to split the sample based on eye state AND self-report. Previous notebook only did eye state.  

6/12/26. revising everythign to use mackenzie's csv.

In [30]:
import pandas as pd

df = pd.read_csv(r"C:\Users\tempu\Downloads\research\labs\gratton\Arousal-Project\splitting_sample\full_sample.csv")
df.head(3)

,sub,ses,task,run,filename,number_of_TRs,abnormal_TR,error_run,note,sleepiness__1_7_scale_,...,confidence,alignment,tmask_filepath,tmask_filename,num_tps_retained,percent_tps_retained,fFD_filepath,fFD_filename,mean_fFD,max_fFD
0,sub-INET001,ses-1,task-rest,run-01,sub-INET001_ses-1_task-rest_run-01_bold.nii.gz,270,0,NaN,NaN,1.0,...,NaN,include,/gpfs/research/grattonlab/iNetworks/Nifti/deri...,sub-INET001_ses-1_task-rest_run-01_desc-tmask_...,263,97.407407,/gpfs/research/grattonlab/iNetworks/Nifti/deri...,sub-INET001_ses-1_task-rest_run-01_desc-fFD.txt,0.026217,0.166371
1,sub-INET001,ses-1,task-rest,run-02,sub-INET001_ses-1_task-rest_run-02_bold.nii.gz,270,0,NaN,NaN,1.0,...,NaN,include,/gpfs/research/grattonlab/iNetworks/Nifti/deri...,sub-INET001_ses-1_task-rest_run-02_desc-tmask_...,265,98.148148,/gpfs/research/grattonlab/iNetworks/Nifti/deri...,sub-INET001_ses-1_task-rest_run-02_desc-fFD.txt,0.024525,0.084208
2,sub-INET001,ses-1,task-rest,run-03,sub-INET001_ses-1_task-rest_run-03_bold.nii.gz,270,0,NaN,NaN,1.0,...,NaN,include,/gpfs/research/grattonlab/iNetworks/Nifti/deri...,sub-INET001_ses-1_task-rest_run-03_desc-tmask_...,265,98.148148,/gpfs/research/grattonlab/iNetworks/Nifti/deri...,sub-INET001_ses-1_task-rest_run-03_desc-fFD.txt,0.024502,0.093614


In [31]:
df.shape

(2474, 22)

In [32]:
df.columns

Index(['sub', 'ses', 'task', 'run', 'filename', 'number_of_TRs', 'abnormal_TR',
       'error_run', 'note', 'sleepiness__1_7_scale_', 'alertness', 'focus',
       'confidence', 'alignment', 'tmask_filepath', 'tmask_filename',
       'num_tps_retained', 'percent_tps_retained', 'fFD_filepath',
       'fFD_filename', 'mean_fFD', 'max_fFD'],
      dtype='object')

In [33]:
len(df["sub"].unique())

94

94 subjects total in the csv.

In [34]:
df["note"].unique()

array([nan, 'done after MRA/MRV', 'big spike at end', 'long blinks',
       '1st time stopped early, restarted', 'slow blinks',
       'HIRMM 81% <.1 mm', '1st 30 sec have a lot of movement',
       'really great, surprisingly', 'had to restart', 'mask',
       'new field map', 'field map 1', 'field map 2', 'fieldmap 1',
       'fieldmap 2', 'stopped in middle, got dizzy, squeezed ball',
       'super long blinks', 'long blinks in last 90 seconds',
       'long blinks in second half', 'long blinks in last minute',
       'long blinks; especially in the last minute',
       'long blinks in 2nd half', 'lots of movement',
       'head moved location', 'lots of motion', 'pulse stopped',
       'high motion', 'motion in second half', 'motion',
       'resp belt stopped', 'long blinks, respiratory belt stopped',
       'long blinks first half', 'Long blinks throughout', 'Long blinks',
       'longer blinks, respiration belt stopped',
       'had to restart --> 2nd scout and fieldmap starts h

In [35]:
# Need to re-make eyes_closed category

eyes_closed_notes = ['long blinks', 'slow blinks', 'super long blinks', 
       'long blinks in last 90 seconds',
       'long blinks in second half',
       'long blinks; especially in the last minute',
       'long blinks in 2nd half',
       'long blinks, respiratory belt stopped',
       'long blinks first half', 'Long blinks throughout', 'Long blinks',
       'longer blinks, respiration belt stopped',
       'long blinks throughout',
       'long blinks in the 2nd half',
       'lots of long blinks',
       'long eye blinks', 'eyes/long blinks',
       ]

import numpy as np

for i in range(len(df)):
    if df.loc[i, "note"] in eyes_closed_notes:
        df.loc[i, "category"] = "eyes_closed"

df.rename(columns = {"sleepiness__1_7_scale_" : "sleep-score", "percent_tps_retained" : "FD-hold"}, inplace = True)
df["FD-hold"] = df["FD-hold"] * .01
len(df[df["category"] == "eyes_closed"])

34

*(IGNORE. FDs were NaN in old df.)*  

So FDs are the only issue. I don't think there's anything I can do about that though. But I can check. I'll make a list of the subjects I need to check for FD outputs.

In [36]:
#df_fd = df[df['FD-hold'].isna()]
#df_fd = df_fd.groupby('subject').agg('count').reset_index()
#df_fd.subject

6/9. I'm checking the Sharepoint. Many of these FDs do exist. I can add them manually.  

6/10. I added them manually.

In [37]:
#df = pd.read_csv("C:\\Users\\tempu\\Downloads\\research\\labs\\gratton\\Arousal-Project\\splitting_sample\\iNETs-sleep-data.csv")
#df

In [38]:
## CLEANING

#df = df.drop("note", axis=1)

# Drop the category == movement.
# df = df[df['category'] != 'movement']

# Convert FD-hold to numeric.
# df['FD-hold'] = df['FD-hold'].str.replace('\ufeff', '')
# df["FD-hold"] = df["FD-hold"].astype(float)

# len(df[df['sleep-score'].isna()]), len(df[df['FD-hold'].isna()]), len(df[df['category'].isna()])


In [39]:
df["category"].value_counts()

category
eyes_closed    34
Name: count, dtype: int64

We need to add the rest of the eyes_closed runs.

In [40]:
# Clean df_all_cat for merging

df_all_cat = pd.read_csv(r"C:\Users\tempu\Downloads\research\labs\gratton\Arousal-Project\splitting_sample\larger_sample.csv")

df_all_cat["subject"] = "sub-" + df_all_cat["subject"]
df_all_cat["session"] = "ses-" + df_all_cat["session"].astype(str)
df_all_cat["run"] = "run-" + df_all_cat["run"].astype(str)

df_all_cat = df_all_cat.rename(columns = {"subject" : "sub", "session" : "ses", "run" : "run"})


df_all_cat.head()

,file,sub,ses,task,run,sleep-score,FD-hold,category,time,arousal_group
0,sub-01_ses-01_task-resting_bold.nii.gz,sub-1,ses-1,rest,run-nan,1.0,0.994555,NaN,10.8504,NaN
1,sub-01_ses-02_task-resting_bold.nii.gz,sub-1,ses-2,rest,run-nan,5.0,0.995463,NaN,10.8603,NaN
2,sub-01_ses-03_task-resting_bold.nii.gz,sub-1,ses-3,rest,run-nan,5.0,0.995463,NaN,10.8603,NaN
3,sub-01_ses-04_task-resting_bold.nii.gz,sub-1,ses-4,rest,run-nan,4.0,0.984574,NaN,10.7415,NaN
4,sub-01_ses-05_task-resting_bold.nii.gz,sub-1,ses-5,rest,run-nan,2.0,0.992740,NaN,10.8306,NaN


In [41]:
# Merging the missing eyes_closed runs from the larger sample to the full sample.
keys = ['sub', 'ses', 'run']

missing = (
    df_all_cat[df_all_cat['category'] == 'eyes_closed']
    .merge(df[keys].drop_duplicates(), on=keys, how='left', indicator=True)
)
missing = missing[missing['_merge'] == 'left_only'].drop(columns=['_merge'])

df = pd.concat([df, missing], ignore_index=True, sort=False)
df["category"].value_counts()

category
eyes_closed    184
Name: count, dtype: int64

6/12, ADD OTHER CATRGORY VALUESSS  
6/20. What other category values? 184 should be all of them.

In [42]:
df.columns

Index(['sub', 'ses', 'task', 'run', 'filename', 'number_of_TRs', 'abnormal_TR',
       'error_run', 'note', 'sleep-score', 'alertness', 'focus', 'confidence',
       'alignment', 'tmask_filepath', 'tmask_filename', 'num_tps_retained',
       'FD-hold', 'fFD_filepath', 'fFD_filename', 'mean_fFD', 'max_fFD',
       'category', 'file', 'time', 'arousal_group'],
      dtype='object')

In [43]:
df.head(1)

,sub,ses,task,run,filename,number_of_TRs,abnormal_TR,error_run,note,sleep-score,...,num_tps_retained,FD-hold,fFD_filepath,fFD_filename,mean_fFD,max_fFD,category,file,time,arousal_group
0,sub-INET001,ses-1,task-rest,run-01,sub-INET001_ses-1_task-rest_run-01_bold.nii.gz,270.0,0.0,NaN,NaN,1.0,...,263.0,0.974074,/gpfs/research/grattonlab/iNetworks/Nifti/deri...,sub-INET001_ses-1_task-rest_run-01_desc-fFD.txt,0.026217,0.166371,NaN,NaN,NaN,NaN


In [44]:
INET_TR = 1.1
INET_VOLS = 270

df["time"] = (df["num_tps_retained"] * INET_TR) / 60
df

,sub,ses,task,run,filename,number_of_TRs,abnormal_TR,error_run,note,sleep-score,...,num_tps_retained,FD-hold,fFD_filepath,fFD_filename,mean_fFD,max_fFD,category,file,time,arousal_group
0,sub-INET001,ses-1,task-rest,run-01,sub-INET001_ses-1_task-rest_run-01_bold.nii.gz,270.0,0.0,NaN,NaN,1.0,...,263.0,0.974074,/gpfs/research/grattonlab/iNetworks/Nifti/deri...,sub-INET001_ses-1_task-rest_run-01_desc-fFD.txt,0.026217,0.166371,NaN,NaN,4.821667,NaN
1,sub-INET001,ses-1,task-rest,run-02,sub-INET001_ses-1_task-rest_run-02_bold.nii.gz,270.0,0.0,NaN,NaN,1.0,...,265.0,0.981481,/gpfs/research/grattonlab/iNetworks/Nifti/deri...,sub-INET001_ses-1_task-rest_run-02_desc-fFD.txt,0.024525,0.084208,NaN,NaN,4.858333,NaN
2,sub-INET001,ses-1,task-rest,run-03,sub-INET001_ses-1_task-rest_run-03_bold.nii.gz,270.0,0.0,NaN,NaN,1.0,...,265.0,0.981481,/gpfs/research/grattonlab/iNetworks/Nifti/deri...,sub-INET001_ses-1_task-rest_run-03_desc-fFD.txt,0.024502,0.093614,NaN,NaN,4.858333,NaN
3,sub-INET001,ses-1,task-rest,run-04,sub-INET001_ses-1_task-rest_run-04_bold.nii.gz,270.0,0.0,NaN,NaN,1.0,...,265.0,0.981481,/gpfs/research/grattonlab/iNetworks/Nifti/deri...,sub-INET001_ses-1_task-rest_run-04_desc-fFD.txt,0.021245,0.075994,NaN,NaN,4.858333,NaN
4,sub-INET001,ses-1,task-rest,run-05,sub-INET001_ses-1_task-rest_run-05_bold.nii.gz,270.0,0.0,NaN,NaN,1.0,...,265.0,0.981481,/gpfs/research/grattonlab/iNetworks/Nifti/deri...,sub-INET001_ses-1_task-rest_run-05_desc-fFD.txt,0.019148,0.078931,NaN,NaN,4.858333,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2619,sub-INET003,ses-4,rest,run-3.0,NaN,NaN,NaN,NaN,NaN,3.0,...,NaN,0.981481,NaN,NaN,NaN,NaN,eyes_closed,sub-INET003_ses-4_task-rest_run-03_bold.nii.gz,NaN,low
2620,sub-INET003,ses-4,rest,run-4.0,NaN,NaN,NaN,NaN,NaN,4.0,...,NaN,0.959259,NaN,NaN,NaN,NaN,eyes_closed,sub-INET003_ses-4_task-rest_run-04_bold.nii.gz,NaN,low
2621,sub-INET003,ses-4,rest,run-5.0,NaN,NaN,NaN,NaN,NaN,4.0,...,NaN,0.974074,NaN,NaN,NaN,NaN,eyes_closed,sub-INET003_ses-4_task-rest_run-05_bold.nii.gz,NaN,low
2622,sub-INET003,ses-4,rest,run-7.0,NaN,NaN,NaN,NaN,NaN,4.0,...,NaN,0.974074,NaN,NaN,NaN,NaN,eyes_closed,sub-INET003_ses-4_task-rest_run-07_bold.nii.gz,NaN,low


### Low Arousal Group

In [45]:
df.columns

Index(['sub', 'ses', 'task', 'run', 'filename', 'number_of_TRs', 'abnormal_TR',
       'error_run', 'note', 'sleep-score', 'alertness', 'focus', 'confidence',
       'alignment', 'tmask_filepath', 'tmask_filename', 'num_tps_retained',
       'FD-hold', 'fFD_filepath', 'fFD_filename', 'mean_fFD', 'max_fFD',
       'category', 'file', 'time', 'arousal_group'],
      dtype='object')

In [46]:
# Split the data into high and low arousal based on sleep-score and category.

df_low = df[((df["sleep-score"] > 5) | (df["category"] == "eyes_closed"))]

# Group by subject and aggregate by sum for time.
cols = ['ses', 'task', 'run', 'filename', 'number_of_TRs', 'abnormal_TR',
       'error_run', 'sleep-score', 'alertness', 'focus', 'confidence',
       'alignment', 'tmask_filepath', 'tmask_filename', 'num_tps_retained',
       'FD-hold', 'fFD_filepath', 'fFD_filename', 'mean_fFD', 'max_fFD',
       'category']
df_low_time = df_low.drop(cols, axis=1)
df_low_time = df_low_time.groupby("sub").agg({"time": "sum"})

# Fill in the subject's time for all runs.
df_low["total_time"] = df_low["sub"].map(df_low_time["time"])

# For merging
df_low["arousal_group"] = "low"

df_low.nlargest(10, "total_time")

C:\Users\tempu\AppData\Local\Temp\ipykernel_32296\871786730.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_low["total_time"] = df_low["sub"].map(df_low_time["time"])
C:\Users\tempu\AppData\Local\Temp\ipykernel_32296\871786730.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_low["arousal_group"] = "low"


,sub,ses,task,run,filename,number_of_TRs,abnormal_TR,error_run,note,sleep-score,...,FD-hold,fFD_filepath,fFD_filename,mean_fFD,max_fFD,category,file,time,arousal_group,total_time
1433,sub-INET077,ses-1,task-rest,run-02,sub-INET077_ses-1_task-rest_run-02_bold.nii.gz,270.0,0.0,NaN,long blinks in last 90 seconds,2.0,...,0.922222,/gpfs/research/grattonlab/iNetworks/Nifti/deri...,sub-INET077_ses-1_task-rest_run-02_desc-fFD.txt,0.037308,0.279164,eyes_closed,NaN,4.565000,low,20.075
1434,sub-INET077,ses-1,task-rest,run-03,sub-INET077_ses-1_task-rest_run-03_bold.nii.gz,270.0,0.0,NaN,long blinks in second half,2.0,...,0.929630,/gpfs/research/grattonlab/iNetworks/Nifti/deri...,sub-INET077_ses-1_task-rest_run-03_desc-fFD.txt,0.039558,0.311736,eyes_closed,NaN,4.601667,low,20.075
1438,sub-INET077,ses-1,task-rest,run-07,sub-INET077_ses-1_task-rest_run-07_bold.nii.gz,270.0,0.0,NaN,long blinks,3.0,...,0.603704,/gpfs/research/grattonlab/iNetworks/Nifti/deri...,sub-INET077_ses-1_task-rest_run-07_desc-fFD.txt,0.092414,0.367156,eyes_closed,NaN,2.988333,low,20.075
1442,sub-INET077,ses-2,task-rest,run-05,sub-INET077_ses-2_task-rest_run-05_bold.nii.gz,270.0,0.0,NaN,long blinks; especially in the last minute,2.0,...,0.807407,/gpfs/research/grattonlab/iNetworks/Nifti/deri...,sub-INET077_ses-2_task-rest_run-05_desc-fFD.txt,0.053057,0.289246,eyes_closed,NaN,3.996667,low,20.075
1445,sub-INET077,ses-3,task-rest,run-02,sub-INET077_ses-3_task-rest_run-02_bold.nii.gz,270.0,0.0,NaN,long blinks in 2nd half,2.0,...,0.792593,/gpfs/research/grattonlab/iNetworks/Nifti/deri...,sub-INET077_ses-3_task-rest_run-02_desc-fFD.txt,0.053798,0.480189,eyes_closed,NaN,3.923333,low,20.075
2600,sub-INET077,ses-3,rest,run-2.0,NaN,NaN,NaN,NaN,NaN,2.0,...,0.788889,NaN,NaN,NaN,NaN,eyes_closed,sub-INET077_ses-3_task-rest_run-02_bold.nii.gz,NaN,low,20.075
2601,sub-INET077,ses-3,rest,run-3.0,NaN,NaN,NaN,NaN,NaN,2.0,...,0.840741,NaN,NaN,NaN,NaN,eyes_closed,sub-INET077_ses-3_task-rest_run-03_bold.nii.gz,NaN,low,20.075
2602,sub-INET077,ses-2,rest,run-1.0,NaN,NaN,NaN,NaN,NaN,2.0,...,0.818519,NaN,NaN,NaN,NaN,eyes_closed,sub-INET077_ses-2_task-rest_run-01_bold.nii.gz,NaN,low,20.075
2603,sub-INET077,ses-2,rest,run-5.0,NaN,NaN,NaN,NaN,NaN,2.0,...,0.814815,NaN,NaN,NaN,NaN,eyes_closed,sub-INET077_ses-2_task-rest_run-05_bold.nii.gz,NaN,low,20.075
2604,sub-INET077,ses-2,rest,run-6.0,NaN,NaN,NaN,NaN,NaN,2.0,...,0.881481,NaN,NaN,NaN,NaN,eyes_closed,sub-INET077_ses-2_task-rest_run-06_bold.nii.gz,NaN,low,20.075


In [47]:
df_low_group = df_low[df_low["total_time"] >= 5]
df_low_prec = df_low[df_low["total_time"] >= 20]

df_low_group.shape, df_low_prec.shape

((80, 27), (15, 27))

In [48]:
len(df_low_prec["sub"].unique()), len(df_low_group["sub"].unique())

(1, 10)

**Only INET077 is precision now.**

['sub-INET003', 'sub-INET032', 'sub-INET077', 'sub-INET083',
       'sub-INET085', 'sub-INET091', 'sub-INET093', 'sub-INET094',
       'sub-INET143', 'sub-INET158'] are group-level low.

### High Arousal Group

In [49]:
df_high = df[((df["sleep-score"] < 3) | (df["category"] == "eyes_opened"))]

# Group by subject and aggregate by sum for time.
cols = ['file', 'ses', 'task', 'run', 'sleep-score', 'FD-hold', 'category']
df_high_time = df_high.drop(cols, axis=1)
df_high_time = df_high_time.groupby("sub").agg({"time": "sum"})

# Fill in the subject's time for all runs.
df_high["total_time"] = df_high["sub"].map(df_high_time["time"])

# For merging
df_high["arousal_group"] = "high"

df_high.describe()

C:\Users\tempu\AppData\Local\Temp\ipykernel_32296\3303240812.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_high["total_time"] = df_high["sub"].map(df_high_time["time"])
C:\Users\tempu\AppData\Local\Temp\ipykernel_32296\3303240812.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_high["arousal_group"] = "high"


,number_of_TRs,abnormal_TR,error_run,sleep-score,alertness,focus,confidence,num_tps_retained,FD-hold,mean_fFD,max_fFD,time,total_time
count,1179.000000,1179.000000,1.0,1224.000000,0.0,0.0,0.0,1179.000000,1224.000000,1179.000000,1179.000000,1179.000000,1224.000000
mean,269.783715,0.003393,1.0,1.566585,NaN,NaN,NaN,252.428329,0.934525,0.031198,0.294536,4.627853,83.244402
std,7.339213,0.058173,NaN,0.495543,NaN,NaN,NaN,22.023808,0.077945,0.025029,0.326191,0.403770,34.565566
min,18.000000,0.000000,1.0,1.000000,NaN,NaN,NaN,13.000000,0.525926,0.006783,0.019846,0.238333,0.000000
25%,270.000000,0.000000,1.0,1.000000,NaN,NaN,NaN,250.000000,0.925926,0.016461,0.112186,4.583333,53.258333
50%,270.000000,0.000000,1.0,2.000000,NaN,NaN,NaN,262.000000,0.970370,0.023687,0.189644,4.803333,84.828333
75%,270.000000,0.000000,1.0,2.000000,NaN,NaN,NaN,264.000000,0.977778,0.036873,0.342507,4.840000,110.458333
max,270.000000,1.000000,1.0,2.000000,NaN,NaN,NaN,265.000000,0.981481,0.262828,3.791911,4.858333,140.690000


In [50]:
df_high_group = df_high[df_high["total_time"] >= 5]
df_high_prec = df_high[df_high["total_time"] >= 20]

df_high_group.shape, df_high_prec.shape

len(df_high_prec["sub"].unique()), len(df_high_group["sub"].unique())

(74, 84)

In [51]:
df_high_group["sub"].unique()

array(['sub-INET001', 'sub-INET002', 'sub-INET003', 'sub-INET005',
       'sub-INET006', 'sub-INET010', 'sub-INET016', 'sub-INET018',
       'sub-INET019', 'sub-INET026', 'sub-INET027', 'sub-INET029',
       'sub-INET030', 'sub-INET032', 'sub-INET033', 'sub-INET035',
       'sub-INET036', 'sub-INET038', 'sub-INET039', 'sub-INET040',
       'sub-INET042', 'sub-INET044', 'sub-INET045', 'sub-INET046',
       'sub-INET047', 'sub-INET049', 'sub-INET050', 'sub-INET051',
       'sub-INET055', 'sub-INET056', 'sub-INET057', 'sub-INET058',
       'sub-INET060', 'sub-INET061', 'sub-INET062', 'sub-INET063',
       'sub-INET065', 'sub-INET067', 'sub-INET068', 'sub-INET070',
       'sub-INET071', 'sub-INET072', 'sub-INET073', 'sub-INET074',
       'sub-INET075', 'sub-INET077', 'sub-INET083', 'sub-INET084',
       'sub-INET085', 'sub-INET086', 'sub-INET087', 'sub-INET088',
       'sub-INET091', 'sub-INET093', 'sub-INET094', 'sub-INET095',
       'sub-INET096', 'sub-INET099', 'sub-INET101', 'sub-INET1

precision: ['sub-INET001', 'sub-INET002', 'sub-INET003', 'sub-INET005',
       'sub-INET006', 'sub-INET010', 'sub-INET018', 'sub-INET019',
       'sub-INET027', 'sub-INET029', 'sub-INET030', 'sub-INET032',
       'sub-INET033', 'sub-INET035', 'sub-INET036', 'sub-INET039',
       'sub-INET040', 'sub-INET042', 'sub-INET044', 'sub-INET045',
       'sub-INET047', 'sub-INET049', 'sub-INET050', 'sub-INET051',
       'sub-INET055', 'sub-INET056', 'sub-INET057', 'sub-INET058',
       'sub-INET061', 'sub-INET062', 'sub-INET063', 'sub-INET065',
       'sub-INET067', 'sub-INET068', 'sub-INET070', 'sub-INET072',
       'sub-INET073', 'sub-INET074', 'sub-INET075', 'sub-INET077',
       'sub-INET083', 'sub-INET084', 'sub-INET085', 'sub-INET086',
       'sub-INET088', 'sub-INET091', 'sub-INET093', 'sub-INET094',
       'sub-INET095', 'sub-INET096', 'sub-INET099', 'sub-INET101',
       'sub-INET103', 'sub-INET104', 'sub-INET106', 'sub-INET107',
       'sub-INET108', 'sub-INET109', 'sub-INET110', 'sub-INET112',
       'sub-INET115', 'sub-INET123', 'sub-INET131', 'sub-INET133',
       'sub-INET136', 'sub-INET137', 'sub-INET140', 'sub-INET141',
       'sub-INET143', 'sub-INET156', 'sub-INET158', 'sub-INET159',
       'sub-INET165', 'sub-INET168']

group: ['sub-INET001', 'sub-INET002', 'sub-INET003', 'sub-INET005',
       'sub-INET006', 'sub-INET010', 'sub-INET016', 'sub-INET018',
       'sub-INET019', 'sub-INET026', 'sub-INET027', 'sub-INET029',
       'sub-INET030', 'sub-INET032', 'sub-INET033', 'sub-INET035',
       'sub-INET036', 'sub-INET038', 'sub-INET039', 'sub-INET040',
       'sub-INET042', 'sub-INET044', 'sub-INET045', 'sub-INET046',
       'sub-INET047', 'sub-INET049', 'sub-INET050', 'sub-INET051',
       'sub-INET055', 'sub-INET056', 'sub-INET057', 'sub-INET058',
       'sub-INET060', 'sub-INET061', 'sub-INET062', 'sub-INET063',
       'sub-INET065', 'sub-INET067', 'sub-INET068', 'sub-INET070',
       'sub-INET071', 'sub-INET072', 'sub-INET073', 'sub-INET074',
       'sub-INET075', 'sub-INET077', 'sub-INET083', 'sub-INET084',
       'sub-INET085', 'sub-INET086', 'sub-INET087', 'sub-INET088',
       'sub-INET091', 'sub-INET093', 'sub-INET094', 'sub-INET095',
       'sub-INET096', 'sub-INET099', 'sub-INET101', 'sub-INET103',
       'sub-INET104', 'sub-INET106', 'sub-INET107', 'sub-INET108',
       'sub-INET109', 'sub-INET110', 'sub-INET112', 'sub-INET114',
       'sub-INET115', 'sub-INET120', 'sub-INET123', 'sub-INET131',
       'sub-INET133', 'sub-INET136', 'sub-INET137', 'sub-INET140',
       'sub-INET141', 'sub-INET143', 'sub-INET156', 'sub-INET158',
       'sub-INET159', 'sub-INET160', 'sub-INET165', 'sub-INET168']

### Finalizing Samples   

## GROUP

In [ ]:
import numpy as np

df_low["analysis_group"] = np.where(
    df_low["total_time"] >= 20,
    "precision",
    np.where(df_low["total_time"] >= 5, "group", "excluded")
)

df_high["analysis_group"] = np.where(
    df_high["total_time"] >= 20,
    "precision",
    np.where(df_high["total_time"] >= 5, "group", "excluded")
)

C:\Users\tempu\AppData\Local\Temp\ipykernel_32296\1382164404.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_low["analysis_group"] = np.where(
C:\Users\tempu\AppData\Local\Temp\ipykernel_32296\1382164404.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_high["analysis_group"] = np.where(


In [56]:
df_low.to_csv("C:\\Users\\tempu\\Downloads\\research\\labs\\gratton\\Arousal-Project\\splitting_sample\\larger_sample\\low_arousal_sample.csv", index=False)
df_high.to_csv("C:\\Users\\tempu\\Downloads\\research\\labs\\gratton\\Arousal-Project\\splitting_sample\\larger_sample\\high_arousal_sample.csv", index=False)